In [1]:
# ==========================================
# CELL: Groundtruth Merge, Scaling & ML Models
# ==========================================
import os
import re
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

BASE_DIR = r"D:\B. Disease Grading"
GROUNDTRUTHS_DIR = os.path.join(BASE_DIR, "2. Groundtruths")

# Load Lesion Features
df_train_feats = pd.read_csv(os.path.join(BASE_DIR, "train_lesion_features.csv"))
df_test_feats = pd.read_csv(os.path.join(BASE_DIR, "test_lesion_features.csv"))

# Load Groundtruth CSVs
gt_files = glob.glob(os.path.join(GROUNDTRUTHS_DIR, "**", "*.csv"), recursive=True)
train_gt_path = [f for f in gt_files if 'train' in os.path.basename(f).lower()][0]
test_gt_path = [f for f in gt_files if 'test' in os.path.basename(f).lower()][0]

df_train_gt = pd.read_csv(train_gt_path)
df_test_gt = pd.read_csv(test_gt_path)

train_img_col = [c for c in df_train_gt.columns if 'image' in c.lower() or 'name' in c.lower()][0]
train_grade_col = [c for c in df_train_gt.columns if 'grade' in c.lower() or 'dr' in c.lower() or 'label' in c.lower()][0]

test_img_col = [c for c in df_test_gt.columns if 'image' in c.lower() or 'name' in c.lower()][0]
test_grade_col = [c for c in df_test_gt.columns if 'grade' in c.lower() or 'dr' in c.lower() or 'label' in c.lower()][0]

def extract_numeric_id(val):
    match = re.search(r'\d+', str(val))
    return int(match.group()) if match else -1

df_train_feats['Image_ID_Num'] = df_train_feats['Image_Name'].apply(extract_numeric_id)
df_train_gt['Image_ID_Num'] = df_train_gt[train_img_col].apply(extract_numeric_id)

df_test_feats['Image_ID_Num'] = df_test_feats['Image_Name'].apply(extract_numeric_id)
df_test_gt['Image_ID_Num'] = df_test_gt[test_img_col].apply(extract_numeric_id)

# Merge
df_train = pd.merge(df_train_feats, df_train_gt[['Image_ID_Num', train_grade_col]], on='Image_ID_Num', how='inner')
df_test = pd.merge(df_test_feats, df_test_gt[['Image_ID_Num', test_grade_col]], on='Image_ID_Num', how='inner')

feature_cols = ['MA_Count', 'MA_Area', 'Hem_Count', 'Hem_Area', 'Ex_Count', 'Ex_Area', 'GLCM_Contrast', 'GLCM_Homogeneity']

X_train = df_train[feature_cols]
y_train = df_train[train_grade_col]

X_test = df_test[feature_cols]
y_test = df_test[test_grade_col]

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def calculate_specificity(cm):
    specificities = []
    for i in range(len(cm)):
        tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
        fp = np.sum(np.delete(cm, i, axis=0)[:, i])
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(spec)
    return np.mean(specificities)

# 1. Support Vector Machine (RBF Kernel)
svm_model = SVC(kernel='rbf', C=10.0, class_weight='balanced', random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

cm_svm = confusion_matrix(y_test, y_pred_svm)
acc_svm = accuracy_score(y_test, y_pred_svm) * 100
sen_svm = recall_score(y_test, y_pred_svm, average='macro') * 100
spec_svm = calculate_specificity(cm_svm) * 100
f1_svm = f1_score(y_test, y_pred_svm, average='macro') * 100

# 2. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

cm_rf = confusion_matrix(y_test, y_pred_rf)
acc_rf = accuracy_score(y_test, y_pred_rf) * 100
sen_rf = recall_score(y_test, y_pred_rf, average='macro') * 100
spec_rf = calculate_specificity(cm_rf) * 100
f1_rf = f1_score(y_test, y_pred_rf, average='macro') * 100

# Results Summary
results = {
    'Method': ['SVM (Lesion-Based ML)', 'Random Forest (Lesion-Based ML)'],
    'Accuracy (%)': [f"{acc_svm:.2f}", f"{acc_rf:.2f}"],
    'Sensitivity (%)': [f"{sen_svm:.2f}", f"{sen_rf:.2f}"],
    'Specificity (%)': [f"{spec_svm:.2f}", f"{spec_rf:.2f}"],
    'F1-Score (%)': [f"{f1_svm:.2f}", f"{f1_rf:.2f}"]
}

print("\n=== LESION-SEGMENTED TRADITIONAL ML RESULTS ===")
print(pd.DataFrame(results).to_string(index=False))


=== LESION-SEGMENTED TRADITIONAL ML RESULTS ===
                         Method Accuracy (%) Sensitivity (%) Specificity (%) F1-Score (%)
          SVM (Lesion-Based ML)        26.21           34.35           82.18        25.48
Random Forest (Lesion-Based ML)        15.53           12.42           77.77        13.58
